# Toy Retriever — Procurement Documents

A minimal retrieval system: 10 short procurement documents, TF-IDF vectors, cosine similarity ranking.

**Why TF-IDF instead of a neural sentence-embedding model?** This keeps the notebook dependency-free (no model download required), and it's actually *better* for this exercise — TF-IDF fails on paraphrased terminology in a very visible, easy-to-diagnose way. The mechanics are identical to any embedding retriever: encode text as a vector, rank candidates by cosine similarity to the query vector.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import random

documents = [
    # 0 - EMD for T-101
    "Bidders shall submit an Earnest Money Deposit (EMD) of Rs 2,00,000 along with their bid for Tender T-101. The EMD shall be refunded within 30 days of contract award to unsuccessful bidders.",
    # 1 - Bid security for T-102 (same concept as doc 0, different tender, different wording)
    "All bidders participating in Tender T-102 must furnish a Bid Security Deposit of Rs 3,50,000. This deposit will be returned within 45 days after the work order is issued to the successful contractor.",
    # 2 - Eligibility: turnover
    "To be eligible for Tender T-101, the bidder must have an average annual turnover of at least Rs 5 crore over the last three financial years.",
    # 3 - Eligibility: prior experience
    "Bidders for Tender T-101 must have successfully completed at least two similar infrastructure projects in the past five years to qualify.",
    # 4 - Defect liability period
    "The contractor shall be responsible for rectifying any defects in the completed work for a period of 24 months from the date of completion, known as the Defect Liability Period.",
    # 5 - Penalty for delay
    "In case of delay in completion of work beyond the scheduled date, a penalty of 0.5 percent of the contract value shall be levied for each week of delay, up to a maximum of 10 percent.",
    # 6 - Payment milestones
    "Payments to the contractor shall be released in three milestones: 30 percent on mobilization, 40 percent on completion of civil work, and 30 percent on final handover.",
    # 7 - Dispute resolution
    "Any dispute arising out of this contract shall first be referred to mutual negotiation, and if unresolved within 30 days, shall be settled through arbitration under the Arbitration and Conciliation Act.",
    # 8 - Technical evaluation weightage
    "The technical evaluation for Tender T-101 shall be scored out of 100 points, with 40 points for technical capability, 35 points for financial bid, and 25 points for past performance.",
    # 9 - Deadline / addendum
    "The original deadline for bid submission under Tender T-101 was June 30 2026. Per Corrigendum No. 2, this deadline has been extended to July 15 2026.",
]

for i, d in enumerate(documents):
    print(f"[{i}] {d}")


[0] Bidders shall submit an Earnest Money Deposit (EMD) of Rs 2,00,000 along with their bid for Tender T-101. The EMD shall be refunded within 30 days of contract award to unsuccessful bidders.
[1] All bidders participating in Tender T-102 must furnish a Bid Security Deposit of Rs 3,50,000. This deposit will be returned within 45 days after the work order is issued to the successful contractor.
[2] To be eligible for Tender T-101, the bidder must have an average annual turnover of at least Rs 5 crore over the last three financial years.
[3] Bidders for Tender T-101 must have successfully completed at least two similar infrastructure projects in the past five years to qualify.
[4] The contractor shall be responsible for rectifying any defects in the completed work for a period of 24 months from the date of completion, known as the Defect Liability Period.
[5] In case of delay in completion of work beyond the scheduled date, a penalty of 0.5 percent of the contract value shall be levied 

## Build the index

Fit one TF-IDF vectorizer on all 10 documents. Every query gets projected into that same vector space, then ranked by cosine similarity against the 10 document vectors.

In [2]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(documents)

def search(query, top_k=5):
    qvec = vectorizer.transform([query])
    sims = cosine_similarity(qvec, doc_vectors)[0]
    ranked = np.argsort(-sims)[:top_k]
    return [(int(i), float(sims[i])) for i in ranked]

def show(query, intended_doc=None, top_k=5):
    print(f"Q: {query}\n")
    results = search(query, top_k=top_k)
    for rank, (i, score) in enumerate(results, start=1):
        flag = "  <-- intended" if i == intended_doc else ""
        print(f"  rank {rank} | doc {i} | score {score:.3f}{flag}")
        print(f"           {documents[i][:80]}...")
    if intended_doc is not None:
        ranks = [i for i, _ in results]
        if intended_doc in ranks:
            print(f"\n  intended doc {intended_doc} landed at rank {ranks.index(intended_doc)+1}")
        else:
            print(f"\n  intended doc {intended_doc} did NOT appear in the top {top_k}")
    print()


## Step 2 — 5 random questions, judge for yourself

Below is a bank of 10 candidate questions, each paired with the document it's *meant* to retrieve. Every time you run the next cell, `random.sample` picks a fresh 5 — no cherry-picking.

For each one, look at the ranked output and judge:
- Did the right document come back **at all**?
- Did it land in the **top 1–2**, or was it buried lower?

Don't peek at the "intended doc" flag until after you've formed your own judgment from the ranked list.

In [3]:
question_bank = {
    "What is the EMD amount required for Tender T-101?": 0,
    "How much bid security deposit must be submitted for Tender T-102?": 1,
    "What annual turnover is required to qualify for Tender T-101?": 2,
    "How many similar projects must a bidder have completed to be eligible for Tender T-101?": 3,
    "How long is the defect liability period after project completion?": 4,
    "What penalty applies if the contractor delays completion of the work?": 5,
    "What are the payment milestones for the contractor?": 6,
    "How are disputes under the contract resolved?": 7,
    "What weightage is given to technical capability in the evaluation?": 8,
    "Has the bid submission deadline for Tender T-101 changed?": 9,
}

random.seed()  # no fixed seed -- genuinely different 5 each run
picked = random.sample(list(question_bank.items()), 5)

for q, intended in picked:
    show(q, intended_doc=intended, top_k=3)


Q: What penalty applies if the contractor delays completion of the work?

  rank 1 | doc 5 | score 0.299  <-- intended
           In case of delay in completion of work beyond the scheduled date, a penalty of 0...
  rank 2 | doc 4 | score 0.248
           The contractor shall be responsible for rectifying any defects in the completed ...
  rank 3 | doc 6 | score 0.229
           Payments to the contractor shall be released in three milestones: 30 percent on ...

  intended doc 5 landed at rank 1

Q: Has the bid submission deadline for Tender T-101 changed?

  rank 1 | doc 9 | score 0.572  <-- intended
           The original deadline for bid submission under Tender T-101 was June 30 2026. Pe...
  rank 2 | doc 0 | score 0.141
           Bidders shall submit an Earnest Money Deposit (EMD) of Rs 2,00,000 along with th...
  rank 3 | doc 3 | score 0.126
           Bidders for Tender T-101 must have successfully completed at least two similar i...

  intended doc 9 landed at rank 1

Q: How m

**Your judgment** (fill in after reading the output above):

| Question | Right doc returned? | Rank (1-2, or buried)? |
|---|---|---|
| 1. | | |
| 2. | | |
| 3. | | |
| 4. | | |
| 5. | | |


## Step 3 — Now let's deliberately break it

Two classic ways to break a lexical retriever like this:

1. **Paraphrased terminology** — ask using words that mean the same thing but don't appear in the source text.
2. **A query that legitimately matches two documents almost equally** — when the corpus has near-duplicate content (e.g. two tenders both requiring a "deposit", under different names and amounts).

Try the two attacks below, then try constructing your own against any document above.

In [4]:
# Attack 1: paraphrased terminology
# The source text (doc 5) says "penalty... percent of the contract value... delay"
# This query says the same thing using none of those words.
show("What financial guarantee must a contractor pay if work is late?", intended_doc=5, top_k=5)


Q: What financial guarantee must a contractor pay if work is late?

  rank 1 | doc 4 | score 0.194
           The contractor shall be responsible for rectifying any defects in the completed ...
  rank 2 | doc 2 | score 0.188
           To be eligible for Tender T-101, the bidder must have an average annual turnover...
  rank 3 | doc 1 | score 0.184
           All bidders participating in Tender T-102 must furnish a Bid Security Deposit of...
  rank 4 | doc 6 | score 0.179
           Payments to the contractor shall be released in three milestones: 30 percent on ...
  rank 5 | doc 8 | score 0.100
           The technical evaluation for Tender T-101 shall be scored out of 100 points, wit...

  intended doc 5 did NOT appear in the top 5



In [5]:
# Attack 2: near-equal match between two near-duplicate documents
# Doc 0 (EMD, Tender T-101) and Doc 1 (Bid Security Deposit, Tender T-102)
# both describe a refundable deposit submitted with a bid -- just for different tenders.
show("How much deposit do bidders need to pay along with their bid?", intended_doc=0, top_k=3)


Q: How much deposit do bidders need to pay along with their bid?

  rank 1 | doc 1 | score 0.452
           All bidders participating in Tender T-102 must furnish a Bid Security Deposit of...
  rank 2 | doc 0 | score 0.397  <-- intended
           Bidders shall submit an Earnest Money Deposit (EMD) of Rs 2,00,000 along with th...
  rank 3 | doc 3 | score 0.146
           Bidders for Tender T-101 must have successfully completed at least two similar i...

  intended doc 0 landed at rank 2



## Construct your own breaking query

Pick any document above and write a question about it that:
- uses a *different word* for the key term than the document does, or
- could plausibly describe two documents at once.

Run it through `show("your question", intended_doc=N)` and see what happens.

In [6]:
# Try your own here
# show("your question", intended_doc=___, top_k=5)
